# ROBUST04 QLD + Cross-Encoder Reranking

**Final Project Part A - Run 3 (Advanced Method)**

**Two-Stage Retrieval:**
1. **Stage 1**: Query Likelihood Dirichlet (QLD) retrieves 1000 docs
2. **Stage 2**: BGE Cross-Encoder reranks top-100

**Why this works:**
- QLD: Better than BM25 for verbose queries
- Cross-Encoder: SOTA semantic matching (contextual understanding)
- Combines statistical (QLD) + neural (BGE) signals

**Expected gain:** 10-20% over BM25 baseline

## 1. Configuration

In [0]:
# Uninstall conflicting packages
# %pip uninstall -y numpy faiss-cpu pyserini sentence-transformers huggingface-hub

# Install compatible versions in correct order
# %pip install numpy==1.24.3 -q
%pip install faiss-cpu -q #==1.7.4 -q
%pip install huggingface-hub #==0.20.0 -q
%pip install sentence-transformers #==2.2.2 -q
%pip install pyserini==0.21.0 -q
# %pip install torch==2.0.1 -q

# Restart Python
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/3a/d0/edc009c27b406c4f9cbc79274d6e46d634d139075492ad055e3d68445925/numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/61.0 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/18.3 MB ? eta -:--:--
   ━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/18.3 MB 58.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 9.3/18.3 MB 135.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━ 15.9/18.3 MB 199.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 18.2/18.3 MB 

In [0]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# File paths
QUERIES_PATH = "queriesROBUST.txt"
QRELS_PATH = "qrels_50_Queries"
OUTPUT_DIR = "runs/"

# QLD parameter tuning range
QLD_MU_RANGE = [500, 1000, 1500, 2000, 2500, 3000]

# Default QLD parameter
QLD_MU_DEFAULT = 1000

# Cross-encoder configuration
CROSS_ENCODER_MODEL = 'BAAI/bge-reranker-base'
RERANK_TOP_K = 100  # Rerank top-100 docs
CROSS_ENCODER_MAX_LENGTH = 512  # Max tokens for query-doc pair

# Retrieval configuration
RETRIEVAL_K = 1000  # Initial retrieval
FINAL_K = 1000  # Final output
RUN_TAG = "qld_bge_rerank"

print("✅ Configuration loaded")

✅ Configuration loaded


## 2. Install Dependencies

In [0]:
import subprocess
import os

# Set Java environment
os.environ['JAVA_HOME'] = '/usr/lib/jvm/zulu17-ca-amd64'
os.environ['JVM_PATH'] = '/usr/lib/jvm/zulu17-ca-amd64/lib/server/libjvm.so'

# Verify
result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print("Java version:")
print(result.stderr)

Java version:
openjdk version "1.8.0_462"
OpenJDK Runtime Environment (Zulu 8.88.0.19-CA-linux64) (build 1.8.0_462-b08)
OpenJDK 64-Bit Server VM (Zulu 8.88.0.19-CA-linux64) (build 25.462-b08, mixed mode)



## 3. Java Setup

## 4. Load ROBUST04 Index

**First time setup (one-time only):**
- Downloads and caches index locally (~2GB, takes 5-10 min)
- Subsequent loads are instant!

## 5. Load Cross-Encoder

In [0]:
from sentence_transformers import CrossEncoder
import torch

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load BGE reranker
print(f"Loading {CROSS_ENCODER_MODEL}...")
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, max_length=CROSS_ENCODER_MAX_LENGTH)

print(f"✅ Cross-encoder loaded on {device}")

2026-01-22 13:46:50.427151: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-22 13:46:51.754789: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Device: cuda
Loading BAAI/bge-reranker-base...


/databricks/python/lib/python3.11/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

✅ Cross-encoder loaded on cuda


In [0]:
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import os
import shutil

# Local cache path
LOCAL_INDEX_PATH = "robust04_index"

# Check if index exists locally
print("starting check of LOCAL_INDEX_PATH .....")
if os.path.exists(LOCAL_INDEX_PATH):
    print(f"📂 Loading cached index from {LOCAL_INDEX_PATH}...")
    searcher = LuceneSearcher(LOCAL_INDEX_PATH)
    index_reader = IndexReader(LOCAL_INDEX_PATH)
    print("✅ Loaded from local cache (instant!)")
else:

    print("⬇️ Downloading ROBUST04 index (first time only, ~2GB)...")
    print("   This will take 5-10 minutes...")
    
    # Download from prebuilt
    searcher = LuceneSearcher.from_prebuilt_index('robust04')
    index_reader = IndexReader.from_prebuilt_index('robust04')
    
    # Try to cache for future use
    print(f"\n💾 Caching index to {LOCAL_INDEX_PATH} for future runs...")
    
    try:
        # Pyserini caches to ~/.cache/pyserini/indexes/
        import pyserini
        pyserini_cache = os.path.join(os.path.expanduser('~'), '.cache', 'pyserini', 'indexes')
        
        # Find the robust04 index directory
        if os.path.exists(pyserini_cache):
            for item in os.listdir(pyserini_cache):
                if 'robust04' in item.lower():
                    source_path = os.path.join(pyserini_cache, item)
                    if os.path.isdir(source_path):
                        print(f"   Found index at: {source_path}")
                        shutil.copytree(source_path, LOCAL_INDEX_PATH)
                        print(f"✅ Index cached! Future loads will be instant.")
                        break
        else:
            print("⚠️ Pyserini cache not found, skipping local cache")
    except Exception as e:
        print(f"⚠️ Could not cache index: {e}")
        print("   Index will be re-downloaded next time")

print(f"\n📊 Stats: {index_reader.stats()}")

⬇️ Downloading ROBUST04 index (first time only, ~2GB)...
   This will take 5-10 minutes...


lucene-index.robust04.20221005.252b5e.tar.gz: 1.68GB [00:28, 62.6MB/s]                           



💾 Caching index to robust04_index for future runs...
   Found index at: /root/.cache/pyserini/indexes/lucene-index.robust04.20221005.252b5e.a1abd5437394956b7ec8bea4699b5e46
⚠️ Could not cache index: [Errno 17] File exists: 'robust04_index'
   Index will be re-downloaded next time

📊 Stats: {'total_terms': 174540872, 'documents': 528030, 'non_empty_documents': 528030, 'unique_terms': 923436}


## 6. Load Queries and Qrels

In [0]:
from collections import defaultdict

def load_queries(queries_path):
    """Parse queriesROBUST.txt file."""
    queries = {}
    with open(queries_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t', 1)
            if len(parts) == 2:
                qid = parts[0].strip()
                query_text = parts[1].strip()
                queries[qid] = query_text
    return queries

def load_qrels(qrels_path):
    """Parse qrels file."""
    qrels = defaultdict(dict)
    with open(qrels_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid = parts[0]
                doc_id = parts[2]
                relevance = int(parts[3])
                qrels[qid][doc_id] = relevance
    return qrels

# Load data
all_queries = load_queries(QUERIES_PATH)
qrels = load_qrels(QRELS_PATH)
train_qids = list(qrels.keys())

# Split train/test
train_queries = {qid: all_queries[qid] for qid in train_qids if qid in all_queries}
test_qids = [qid for qid in all_queries.keys() if qid not in train_qids]
test_queries = {qid: all_queries[qid] for qid in test_qids}

print(f"✅ Loaded {len(all_queries)} queries")
print(f"   Train: {len(train_queries)}, Test: {len(test_queries)}")

✅ Loaded 249 queries
   Train: 50, Test: 199


## 7. QLD Retrieval Functions

In [0]:
def retrieve_qld(query_text, mu=1000, k=1000):
    """
    Retrieve using Query Likelihood with Dirichlet smoothing.
    
    Args:
        query_text: Query string
        mu: Dirichlet prior parameter
        k: Number of documents to retrieve
        
    Returns:
        List of (doc_id, score, doc_text) tuples
    """
    # Set QLD parameters
    searcher.set_qld(mu=mu)
    
    # Search
    hits = searcher.search(query_text, k=k)
    
    # Get document text for reranking
    results = []
    for hit in hits:
        try:
            doc = searcher.doc(hit.docid)
            doc_text = doc.raw()  # Get raw document text
            results.append((hit.docid, hit.score, doc_text))
        except:
            # If document not found, use empty text
            results.append((hit.docid, hit.score, ""))
    
    return results

def batch_retrieve_qld(queries_dict, mu, k=1000, desc="Retrieving"):
    """Batch QLD retrieval."""
    from tqdm import tqdm
    
    results = {}
    for qid, query_text in tqdm(queries_dict.items(), desc=desc):
        results[qid] = retrieve_qld(query_text, mu=mu, k=k)
    
    return results

print("✅ QLD retrieval functions ready")

✅ QLD retrieval functions ready


## 8. Cross-Encoder Reranking

In [0]:
import json

def rerank_with_cross_encoder(query_text, initial_results, top_k=100):
    """
    Rerank documents using BGE cross-encoder.
    
    Args:
        query_text: Query string
        initial_results: List of (doc_id, score, doc_text) from QLD
        top_k: Number of top docs to rerank
        
    Returns:
        List of (doc_id, rerank_score) tuples
    """
    if not initial_results:
        return []
    
    # Take top-k results for reranking
    candidates = initial_results[:top_k]
    
    # Prepare query-document pairs
    pairs = []
    doc_ids = []
    
    for doc_id, qld_score, doc_text in candidates:
        # Parse document text (JSON format from Pyserini)
        try:
            doc_data = json.loads(doc_text)
            doc_content = doc_data.get('contents', '')
        except:
            doc_content = doc_text
        
        # Truncate document to avoid token limit issues
        doc_content = doc_content[:2000]  # Roughly 500 tokens
        
        pairs.append([query_text, doc_content])
        doc_ids.append(doc_id)
    
    # Score with cross-encoder
    scores = cross_encoder.predict(pairs)
    
    # Combine doc_ids with scores
    reranked = [(doc_id, float(score)) for doc_id, score in zip(doc_ids, scores)]
    
    # Sort by cross-encoder score (descending)
    reranked.sort(key=lambda x: x[1], reverse=True)
    
    return reranked

def retrieve_qld_and_rerank(query_text, mu, retrieval_k, rerank_top_k, final_k):
    """
    Two-stage retrieval: QLD + Cross-Encoder reranking.
    
    Args:
        query_text: Query string
        mu: QLD Dirichlet parameter
        retrieval_k: Number of docs to retrieve with QLD
        rerank_top_k: Number of top docs to rerank
        final_k: Final number of docs to return
        
    Returns:
        List of (doc_id, final_score) tuples
    """
    # Stage 1: QLD retrieval
    initial_results = retrieve_qld(query_text, mu=mu, k=retrieval_k)
    
    if not initial_results:
        return []
    
    # Stage 2: Rerank top-k
    reranked_results = rerank_with_cross_encoder(
        query_text, 
        initial_results, 
        top_k=rerank_top_k
    )
    
    # Combine: reranked docs + remaining docs (not reranked)
    reranked_doc_ids = {doc_id for doc_id, _ in reranked_results}
    
    # Get remaining docs (those not reranked)
    remaining_results = [
        (doc_id, qld_score) 
        for doc_id, qld_score, _ in initial_results[rerank_top_k:]
    ]
    
    # Final ranking: reranked docs first, then remaining
    final_results = reranked_results + remaining_results
    
    # Return top final_k
    return final_results[:final_k]

def batch_retrieve_qld_rerank(queries_dict, mu, retrieval_k, rerank_top_k, 
                              final_k, desc="Retrieving"):
    """Batch QLD + reranking."""
    from tqdm import tqdm
    
    results = {}
    for qid, query_text in tqdm(queries_dict.items(), desc=desc):
        results[qid] = retrieve_qld_and_rerank(
            query_text, mu, retrieval_k, rerank_top_k, final_k
        )
    
    return results

print("✅ Cross-encoder reranking functions ready")

✅ Cross-encoder reranking functions ready


## 9. MAP Evaluation

In [0]:
def calculate_ap(ranked_list, qrels_for_query):
    """Calculate Average Precision."""
    if not qrels_for_query:
        return 0.0
    
    num_relevant = sum(1 for rel in qrels_for_query.values() if rel > 0)
    if num_relevant == 0:
        return 0.0
    
    num_relevant_retrieved = 0
    precision_sum = 0.0
    
    for i, doc_id in enumerate(ranked_list, start=1):
        if doc_id in qrels_for_query and qrels_for_query[doc_id] > 0:
            num_relevant_retrieved += 1
            precision_at_i = num_relevant_retrieved / i
            precision_sum += precision_at_i
    
    if num_relevant_retrieved == 0:
        return 0.0
    
    return precision_sum / num_relevant

def calculate_map(retrieval_results, qrels):
    """Calculate Mean Average Precision."""
    ap_scores = []
    
    for qid in retrieval_results:
        if qid not in qrels:
            continue
        
        ranked_list = [doc_id for doc_id, _ in retrieval_results[qid]]
        ap = calculate_ap(ranked_list, qrels[qid])
        ap_scores.append(ap)
    
    if not ap_scores:
        return 0.0
    
    return sum(ap_scores) / len(ap_scores)

print("✅ MAP evaluation ready")

✅ MAP evaluation ready


## 10. Test QLD Only (No Reranking)

In [0]:
print("🧪 Testing QLD baseline (no reranking)...\n")

# Test different mu values
qld_baseline_results = {}

for mu in [500, 1000, 2000]:
    results = batch_retrieve_qld(
        train_queries,
        mu=mu,
        k=RETRIEVAL_K,
        desc=f"QLD mu={mu}"
    )
    
    # Convert to (doc_id, score) format for MAP calculation
    results_for_map = {
        qid: [(doc_id, score) for doc_id, score, _ in docs]
        for qid, docs in results.items()
    }
    
    map_score = calculate_map(results_for_map, qrels)
    qld_baseline_results[mu] = map_score
    print(f"QLD mu={mu}: MAP={map_score:.4f}")

best_mu_baseline = max(qld_baseline_results, key=qld_baseline_results.get)
print(f"\n✅ Best QLD baseline: mu={best_mu_baseline}, MAP={qld_baseline_results[best_mu_baseline]:.4f}")

🧪 Testing QLD baseline (no reranking)...



QLD mu=500: 100%|██████████| 50/50 [00:10<00:00,  4.92it/s]


QLD mu=500: MAP=0.2424


QLD mu=1000: 100%|██████████| 50/50 [00:08<00:00,  5.73it/s]


QLD mu=1000: MAP=0.2352


QLD mu=2000: 100%|██████████| 50/50 [00:09<00:00,  5.48it/s]

QLD mu=2000: MAP=0.2261

✅ Best QLD baseline: mu=500, MAP=0.2424


## 11. Test QLD + Reranking

In [0]:
print(f"🧪 Testing QLD + Cross-Encoder reranking...")
print(f"   Reranking top-{RERANK_TOP_K} documents\n")

# Test with best QLD mu
test_results = batch_retrieve_qld_rerank(
    train_queries,
    mu=best_mu_baseline,
    retrieval_k=RETRIEVAL_K,
    rerank_top_k=RERANK_TOP_K,
    final_k=FINAL_K,
    desc="QLD + Reranking"
)

test_map = calculate_map(test_results, qrels)

print(f"\n{'='*60}")
print(f"📊 QLD + RERANKING TEST RESULTS")
print(f"{'='*60}")
print(f"QLD mu: {best_mu_baseline}")
print(f"Rerank top-K: {RERANK_TOP_K}")
print(f"\nPerformance:")
print(f"  QLD only: {qld_baseline_results[best_mu_baseline]:.4f}")
print(f"  QLD + Reranking: {test_map:.4f}")
print(f"  Reranking gain: {((test_map - qld_baseline_results[best_mu_baseline]) / qld_baseline_results[best_mu_baseline] * 100):+.2f}%")
print(f"\nComparison to baselines:")
print(f"  BM25 baseline: 0.2475")
print(f"  BM25+RM3: 0.2713")
print(f"  QLD+Rerank: {test_map:.4f}")
print(f"  Gain over BM25: {((test_map - 0.2475) / 0.2475 * 100):+.2f}%")
print(f"{'='*60}")

🧪 Testing QLD + Cross-Encoder reranking...
   Reranking top-100 documents



QLD + Reranking: 100%|██████████| 50/50 [02:29<00:00,  2.99s/it]


📊 QLD + RERANKING TEST RESULTS
QLD mu: 500
Rerank top-K: 100

Performance:
  QLD only: 0.2424
  QLD + Reranking: 0.2218
  Reranking gain: -8.51%

Comparison to baselines:
  BM25 baseline: 0.2475
  BM25+RM3: 0.2713
  QLD+Rerank: 0.2218
  Gain over BM25: -10.40%


## 12. QLD Parameter Tuning (Optional)

Tune mu parameter with reranking enabled.

In [0]:
print("🔍 Tuning QLD mu parameter with reranking...")

import time
start_time = time.time()

tuning_results = []

for mu in QLD_MU_RANGE:
    results = batch_retrieve_qld_rerank(
        train_queries,
        mu=mu,
        retrieval_k=RETRIEVAL_K,
        rerank_top_k=RERANK_TOP_K,
        final_k=FINAL_K,
        desc=f"mu={mu}"
    )
    
    map_score = calculate_map(results, qrels)
    tuning_results.append({
        'mu': mu,
        'MAP': map_score
    })
    
    print(f"mu={mu}: MAP={map_score:.4f}")

elapsed = time.time() - start_time

# Find best
import pandas as pd
tuning_df = pd.DataFrame(tuning_results).sort_values('MAP', ascending=False)
best_mu = tuning_df.iloc[0]['mu']
best_map = tuning_df.iloc[0]['MAP']

print(f"\n{'='*60}")
print(f"🎯 TUNING RESULTS")
print(f"{'='*60}")
print(f"Best mu: {best_mu}")
print(f"Best MAP: {best_map:.4f}")
print(f"Time: {elapsed:.1f}s")
print(f"{'='*60}\n")

display(tuning_df)

🔍 Tuning QLD mu parameter with reranking...


mu=500: 100%|██████████| 50/50 [02:29<00:00,  3.00s/it]


mu=500: MAP=0.2218


mu=1000: 100%|██████████| 50/50 [02:31<00:00,  3.04s/it]


mu=1000: MAP=0.2194


mu=1500: 100%|██████████| 50/50 [02:32<00:00,  3.04s/it]


... (4 outputs left out)

mu=2500: MAP=0.2115


mu=3000: 100%|██████████| 50/50 [02:32<00:00,  3.06s/it]


mu=3000: MAP=0.2091

🎯 TUNING RESULTS
Best mu: 500.0
Best MAP: 0.2218
Time: 911.9s



mu,MAP
500,0.22176278710817943
1000,0.21944729670845228
1500,0.21838342215101034
2000,0.21477517699443407
2500,0.21152242481498224
3000,0.2091292982711282


## 13. Generate Run 3 - QLD + Cross-Encoder

In [0]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"🚀 Generating Run 3: QLD + Cross-Encoder Reranking")
print(f"   QLD mu: {best_mu}")
print(f"   Rerank top-{RERANK_TOP_K} with {CROSS_ENCODER_MODEL}")
print(f"   Processing all {len(all_queries)} queries...\n")

# Retrieve for all queries
all_results_rerank = batch_retrieve_qld_rerank(
    all_queries,
    mu=best_mu,
    retrieval_k=RETRIEVAL_K,
    rerank_top_k=RERANK_TOP_K,
    final_k=FINAL_K,
    desc="Run 3 retrieval"
)

# Save TREC format
def save_trec_format(retrieval_results, output_path, run_tag):
    """Save results in TREC format."""
    with open(output_path, 'w') as f:
        for qid in sorted(retrieval_results.keys(), key=int):
            ranked_docs = retrieval_results[qid]
            
            for rank, (doc_id, score) in enumerate(ranked_docs, start=1):
                line = f"{qid} Q0 {doc_id} {rank} {score:.6f} {run_tag}\n"
                f.write(line)
    
    print(f"✅ Saved TREC format: {output_path}")

run3_path = os.path.join(OUTPUT_DIR, "run_3.res")
save_trec_format(
    all_results_rerank,
    run3_path,
    f"qld_mu{best_mu}_bge_rerank{RERANK_TOP_K}"
)

# Verify
print(f"\n📊 Run 3 Statistics:")
print(f"   Total queries: {len(all_results_rerank)}")
print(f"   Avg docs per query: {sum(len(docs) for docs in all_results_rerank.values()) / len(all_results_rerank):.1f}")

# Show sample
print(f"\n🔍 Sample output (first 5 lines):")
with open(run3_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(f"   {line.strip()}")

🚀 Generating Run 3: QLD + Cross-Encoder Reranking
   QLD mu: 500.0
   Rerank top-100 with BAAI/bge-reranker-base
   Processing all 249 queries...



Run 3 retrieval: 100%|██████████| 249/249 [12:59<00:00,  3.13s/it]


✅ Saved TREC format: runs/run_3.res

📊 Run 3 Statistics:
   Total queries: 249
   Avg docs per query: 969.2

🔍 Sample output (first 5 lines):
   301 Q0 FBIS4-18122 1 0.944338 qld_mu500.0_bge_rerank100
   301 Q0 FBIS3-19199 2 0.920008 qld_mu500.0_bge_rerank100
   301 Q0 FBIS3-26415 3 0.916435 qld_mu500.0_bge_rerank100
   301 Q0 FBIS3-41247 4 0.908138 qld_mu500.0_bge_rerank100
   301 Q0 FBIS3-23986 5 0.896347 qld_mu500.0_bge_rerank100


## 14. Final Validation & Comparison

In [0]:
# Calculate MAP on train set
train_results_rerank = {qid: all_results_rerank[qid] for qid in train_queries.keys()}
train_map_rerank = calculate_map(train_results_rerank, qrels)

print(f"{'='*60}")
print(f"📈 FINAL COMPARISON (Train Set)")
print(f"{'='*60}")
print(f"Run 1 - BM25:")
print(f"  MAP: 0.2475")
print(f"\nRun 2 - BM25 + RM3:")
print(f"  MAP: 0.2713 (+9.61%)")
print(f"\nRun 3 - QLD + Cross-Encoder:")
print(f"  QLD mu: {best_mu}")
print(f"  Rerank top-{RERANK_TOP_K}")
print(f"  MAP: {train_map_rerank:.4f}")
print(f"  Gain over BM25: {((train_map_rerank - 0.2475) / 0.2475 * 100):+.2f}%")
print(f"  Gain over BM25+RM3: {((train_map_rerank - 0.2713) / 0.2713 * 100):+.2f}%")
print(f"\n🏆 Best Method: Run {[1, 2, 3][max(enumerate([0.2475, 0.2713, train_map_rerank]), key=lambda x: x[1])[0]]}")
print(f"{'='*60}")

📈 FINAL COMPARISON (Train Set)
Run 1 - BM25:
  MAP: 0.2475

Run 2 - BM25 + RM3:
  MAP: 0.2713 (+9.61%)

Run 3 - QLD + Cross-Encoder:
  QLD mu: 500.0
  Rerank top-100
  MAP: 0.2218
  Gain over BM25: -10.40%
  Gain over BM25+RM3: -18.26%

🏆 Best Method: Run 2


## 15. Per-Query Analysis

In [0]:
import pandas as pd

# Per-query performance
per_query_analysis = []

for qid in train_queries.keys():
    ranked_list = [doc_id for doc_id, _ in train_results_rerank[qid]]
    ap = calculate_ap(ranked_list, qrels[qid])
    
    per_query_analysis.append({
        'query_id': qid,
        'query_text': train_queries[qid],
        'AP': ap,
        'num_relevant': sum(1 for rel in qrels[qid].values() if rel > 0)
    })

df_analysis = pd.DataFrame(per_query_analysis).sort_values('AP')

print("📊 Per-Query Analysis: QLD + Cross-Encoder\n")
print("Worst 5 performing queries:")
display(df_analysis.head(5)[['query_id', 'query_text', 'AP']])

print("\nBest 5 performing queries:")
display(df_analysis.tail(5)[['query_id', 'query_text', 'AP']])

print(f"\nAP Statistics:")
print(df_analysis['AP'].describe())

📊 Per-Query Analysis: QLD + Cross-Encoder

Worst 5 performing queries:


query_id,query_text,AP
322,international art crime,7.856418429440743E-4
309,rap crime,0.0010288065843621398
314,marine vegetation,0.0024171683404133194
305,dangerous vehicles,0.004851893513310836
315,unexplained highway accidents,0.01501723521822553



Best 5 performing queries:


query_id,query_text,AP
302,poliomyelitis post polio,0.5557940789504603
316,polygamy polyandry polygyny,0.5754068916840074
348,agoraphobia,0.7100000000000001
313,magnetic levitation maglev,0.8067741313382717
312,hydroponics,0.9215468874559786



AP Statistics:
count    50.000000
mean      0.221763
std       0.224580
min       0.000786
25%       0.051381
50%       0.135507
75%       0.301048
max       0.921547
Name: AP, dtype: float64


## 16. Summary

In [0]:
print(f"""
{'='*60}
🎉 ALL RUNS COMPLETE!
{'='*60}

📊 Final Results Summary:

Run 1 - BM25 (Baseline):
   • Parameters: k1=0.6, b=0.4
   • MAP: 0.2475
   • File: runs/run_1.res

Run 2 - BM25 + RM3 (Query Expansion):
   • RM3: fb_terms=20, fb_docs=15, weight=0.5
   • MAP: 0.2713
   • Gain: +9.61%
   • File: runs/run_2.res

Run 3 - QLD + Cross-Encoder (Neural Reranking):
   • QLD mu: {best_mu}
   • BGE reranker on top-{RERANK_TOP_K}
   • MAP: {train_map_rerank:.4f}
   • Gain over BM25: {((train_map_rerank - 0.2475) / 0.2475 * 100):+.2f}%
   • File: runs/run_3.res

🏆 Best Method: {'Run 3' if train_map_rerank > 0.2713 else 'Run 2'}

📈 Overall Progress:
   Baseline → Best: {((max(train_map_rerank, 0.2713) - 0.2475) / 0.2475 * 100):+.2f}% improvement

📁 Submission Files Ready:
   • runs/run_1.res
   • runs/run_2.res  
   • runs/run_3.res

🎓 Methods Summary:
   1. BM25: Classical probabilistic retrieval
   2. BM25+RM3: Query expansion via pseudo-relevance feedback
   3. QLD+BGE: Neural reranking with SOTA cross-encoder

💡 Key Innovation (Run 3):
   Two-stage architecture combining:
   • Statistical retrieval (QLD) - fast, scalable
   • Neural reranking (BGE) - contextual understanding
   
   This approach balances efficiency with accuracy,
   and is used in production search systems!

{'='*60}
""")


🎉 ALL RUNS COMPLETE!

📊 Final Results Summary:

Run 1 - BM25 (Baseline):
   • Parameters: k1=0.6, b=0.4
   • MAP: 0.2475
   • File: runs/run_1.res

Run 2 - BM25 + RM3 (Query Expansion):
   • RM3: fb_terms=20, fb_docs=15, weight=0.5
   • MAP: 0.2713
   • Gain: +9.61%
   • File: runs/run_2.res

Run 3 - QLD + Cross-Encoder (Neural Reranking):
   • QLD mu: 500.0
   • BGE reranker on top-100
   • MAP: 0.2218
   • Gain over BM25: -10.40%
   • File: runs/run_3.res

🏆 Best Method: Run 2

📈 Overall Progress:
   Baseline → Best: +9.62% improvement

📁 Submission Files Ready:
   • runs/run_1.res
   • runs/run_2.res  
   • runs/run_3.res

🎓 Methods Summary:
   1. BM25: Classical probabilistic retrieval
   2. BM25+RM3: Query expansion via pseudo-relevance feedback
   3. QLD+BGE: Neural reranking with SOTA cross-encoder

💡 Key Innovation (Run 3):
   Two-stage architecture combining:
   • Statistical retrieval (QLD) - fast, scalable
   • Neural reranking (BGE) - contextual understanding
   
   This a

## 17. Cleanup

In [0]:
# Clear GPU memory if used
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Cleanup complete")

✅ Cleanup complete
